In [16]:
from src.feature_engineering import (
    create_features_engineering,
)
from src.dataset_overview import load_data
import pandas as pd
import numpy as np

In [17]:
path = "../data/processed/Cardiovascular_Disease_Dataset_CLEAN.csv"
df = load_data(file_path=path)
df.head()

,patientid,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels,target
0,103368,53,1,2,171,0,0,1,147,0,5.3,3,3,1
1,119250,40,1,0,94,229,0,1,115,0,3.7,1,1,0
2,119372,49,1,2,133,142,0,0,202,1,5.0,1,0,0
3,132514,43,1,0,138,295,1,1,153,0,3.2,2,2,1
4,146211,31,1,1,199,0,0,2,136,0,5.3,3,2,1


In [18]:
df1 = create_features_engineering(df=df,
                                  interactions=[('age','serumcholestrol','mul'),('age','serumcholestrol','add'),('age','serumcholestrol','sub')],
                                  ratios=[('maxheartrate','oldpeak','heardrateandOldpeak')])
df1

,patientid,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels,target,age_mul_serumcholestrol,age_plus_serumcholestrol,age_minus_serumcholestrol,heardrateandOldpeak
0,103368,53,1,2,171,0,0,1,147,0,5.3,3,3,1,0,53,53,27.735849
1,119250,40,1,0,94,229,0,1,115,0,3.7,1,1,0,9160,269,-189,31.081081
2,119372,49,1,2,133,142,0,0,202,1,5.0,1,0,0,6958,191,-93,40.400000
3,132514,43,1,0,138,295,1,1,153,0,3.2,2,2,1,12685,338,-252,47.812500
4,146211,31,1,1,199,0,0,2,136,0,5.3,3,2,1,0,31,31,25.660377
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,9949544,48,1,2,139,349,0,2,183,1,5.6,2,2,1,16752,397,-301,32.678571
996,9953423,47,1,3,143,258,1,1,98,1,5.7,1,0,0,12126,305,-211,17.192982
997,9965859,69,1,0,156,434,1,0,196,0,1.4,3,1,1,29946,503,-365,139.999999
998,9988507,45,1,1,186,417,0,1,117,1,5.9,3,2,1,18765,462,-372,19.830508


In [19]:
df_transaksi = pd.DataFrame({
    "kategori": ["Elektronik", "Elektronik", "Fashion", "Fashion", "Kuliner"],
    "harga_satuan": [1500000, 3000000, 250000, 500000, 50000],
    "qty": [2, 1, 3, 2, 4],
    "diskon": [100000, 250000, 25000, 50000, 0],
    "ongkir": [20000, 35000, 15000, 15000, 10000],
    "tgl_transaksi": ["2026-11-18", "2026-11-21", "2026-11-22", "2026-11-23", "2026-11-24"],
})

In [21]:
def hitung_total_bayar_bersih(df: pd.DataFrame) -> pd.Series:
    return (df["harga_satuan"] * df["qty"]) - df["diskon"] + df["ongkir"]

def kategori_nilai_order(df: pd.DataFrame) -> pd.Series:
    total_kotor = df["harga_satuan"] * df["qty"]
    return pd.Series(np.where(total_kotor >= 1000000, 1, 0), index=df.index)

In [ ]:
df_hasil = df_transaksi.pipe(
    create_features_engineering,
    # A. Interactions: operasi aritmetika antar dua kolom numerik
    interactions=[
        ("harga_satuan", "qty", "mul"),      # menghasilkan: harga_satuan_mul_qty
        ("diskon", "ongkir", "sub"),         # menghasilkan: diskon_minus_ongkir
        ("harga_satuan", "ongkir", "add"),   # menghasilkan: harga_satuan_plus_ongkir
    ],
    # B. Ratios: perbandingan proporsional (aman dari pembagian nol)
    ratios=[
        ("diskon", "harga_satuan", "rasio_diskon_per_unit"),
        ("ongkir", "qty", "ongkir_per_item"),
    ],
    # C. Date Columns: ekstraksi tahun, bulan, tanggal, hari ke-, dan status akhir pekan
    date_cols=["tgl_transaksi"],
    # D. Group Aggregations: statistik agregat berdasarkan kategori tertentu
    group_aggs=[
        {
            "group_col": "kategori",
            "target_col": "harga_satuan",
            "agg": "mean",
            "new_col": "rata_harga_kategori",
        },
        {
            "group_col": "kategori",
            "target_col": "qty",
            "agg": "sum",
        },
    ],
    # E. Custom Functions: aturan bisnis kompleks multi-kolom
    custom_funcs={
        "total_bayar_bersih": hitung_total_bayar_bersih,
        "is_high_value_order": kategori_nilai_order,
    })

df_hasil

,kategori,harga_satuan,qty,diskon,ongkir,tgl_transaksi,harga_satuan_mul_qty,diskon_minus_ongkir,harga_satuan_plus_ongkir,rasio_diskon_per_unit,ongkir_per_item,tgl_transaksi_year,tgl_transaksi_month,tgl_transaksi_day,tgl_transaksi_dayofweek,tgl_transaksi_is_weekend,rata_harga_kategori,qty_sum_by_kategori,total_bayar_bersih,is_high_value_order
0,Elektronik,1500000,2,100000,20000,2026-11-18,3000000,80000,1520000,0.066667,9999.999950,2026,11,18,2,0,2250000.0,3,2920000,1
1,Elektronik,3000000,1,250000,35000,2026-11-21,3000000,215000,3035000,0.083333,34999.999650,2026,11,21,5,1,2250000.0,3,2785000,1
2,Fashion,250000,3,25000,15000,2026-11-22,750000,10000,265000,0.100000,4999.999983,2026,11,22,6,1,375000.0,5,740000,0
3,Fashion,500000,2,50000,15000,2026-11-23,1000000,35000,515000,0.100000,7499.999963,2026,11,23,0,0,375000.0,5,965000,1
4,Kuliner,50000,4,0,10000,2026-11-24,200000,-10000,60000,0.000000,2499.999994,2026,11,24,1,0,50000.0,4,210000,0
